# Acemi Kaggle runner — Stage 6 forward pruning (0.6.0)

Clones the public repository, installs Stockfish and tools, runs the correctness suite, then two matches, and pushes only `data/` records. No analysis. Keep Internet enabled.

1. SPRT: current engine (0.6.0) against the previous version (0.5.0, commit `e985c10`) at 500 ms per move. Only completed colour-swapped pairs count.
2. Rough Elo anchor: 60 games against Stockfish limited to UCI_Elo 1320 at 500 ms per move (a rough scale, not comparable with Lichess or CCRL ratings).

Run records are committed and pushed right after each match, and the last cell pushes anything still unpushed, so one failed push does not lose data.

Before running: create a Kaggle Secret named `GITHUB_TOKEN`. Never paste the token into a cell.

In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "main"
WORKSPACE = "/kaggle/working/acemi"


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def run_quietly(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        details = "\n".join(part for part in (result.stdout, result.stderr) if part)
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{details[-4000:]}")
    return result.stdout

repo_url = GITHUB_REPO_URL.strip()
if not repo_url.startswith("https://github.com/") or "YOUR_ACCOUNT" in repo_url:
    raise ValueError("Set GITHUB_REPO_URL to your repository's HTTPS clone URL.")
if "@" in repo_url.split("github.com/", 1)[-1].split("/", 1)[0]:
    raise ValueError("Do not put credentials in GITHUB_REPO_URL. Use the GITHUB_TOKEN Kaggle Secret.")
workspace = Path(WORKSPACE)
if not workspace.exists():
    run_quietly(["git", "clone", "--branch", GITHUB_BRANCH, repo_url, str(workspace)])
else:
    run_quietly(["git", "-C", str(workspace), "pull", "--ff-only", "origin", GITHUB_BRANCH])
os.chdir(workspace)
print(f"Repository ready on branch {GITHUB_BRANCH}.")


In [ ]:
run_quietly(["apt-get", "update", "-qq"])
run_quietly(["apt-get", "install", "-y", "-qq", "stockfish"])
run_quietly(["python", "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"])
stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    raise RuntimeError("Stockfish package installed but executable was not found.")
version = subprocess.run([stockfish_bin, "--version"], check=True, text=True, capture_output=True).stdout.splitlines()[0]
print(f"Stockfish installed: {version}")
print("Tools installed.")


In [ ]:
# Correctness suite: board, perft, UCI, stats, eval, search.
test_result = subprocess.run(["python", "-m", "unittest", "discover", "-v"], text=True, capture_output=True)
if test_result.stdout:
    print(test_result.stdout, end="")
if test_result.stderr:
    print(test_result.stderr, end="")
if test_result.returncode:
    raise RuntimeError("Correctness suite failed; see test output above.")
print("Correctness suite passed.")


In [ ]:
# Push helper: commits the data/ records and pushes them to GitHub. Used after every match
# and in the last cell. Anything committed earlier but not yet pushed is pushed too.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    try:
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is unavailable; add it under Add-ons â†’ Secrets.") from None
    if not token:
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty.")
    auth_value = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": auth_value,
        "GIT_TERMINAL_PROMPT": "0",
    })

    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=git_env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stderr + result.stdout).replace(token, "[REDACTED]")
            raise RuntimeError(f"Git command failed ({args[0]}): {details[-1500:]}")
        return result

    git("config", "user.name", "Acemi Kaggle Runner")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    git("add", "--", "data/runs", "data/index.csv", "data/positions")
    if git("diff", "--cached", "--quiet", check=False).returncode != 0:
        git("commit", "-m", message)
        print(f"Committed: {message}")
    git("fetch", "origin", GITHUB_BRANCH)
    unpushed = int(git("rev-list", "--count", "FETCH_HEAD..HEAD").stdout.strip())
    if unpushed == 0:
        print("Nothing to push: GitHub already has every local record.")
        return
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()) > 0:
        git("rebase", "FETCH_HEAD")  # GitHub moved on since this session started.
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    local_head = git("rev-parse", "HEAD").stdout.strip()
    remote_line = git("ls-remote", "origin", f"refs/heads/{GITHUB_BRANCH}").stdout.split()
    if not remote_line or remote_line[0] != local_head:
        raise RuntimeError("Push finished but GitHub does not show the new commit; run this cell again.")
    print(f"Pushed to GitHub ({unpushed} commit(s)); {GITHUB_BRANCH} is now at {local_head[:7]}.")

In [ ]:
# SPRT: current engine vs the previous version (0.5.0, commit e985c10), elo0=0 elo1=10, 500 ms per move.
# SPRT stops on a clear pair-complete result; 300 games is the cap (undecided = inconclusive).
OLD_COMMIT = "e985c10"
old_dir = Path("/kaggle/working/acemi_v050")
if not old_dir.exists():
    run_quietly(["git", "clone", repo_url, str(old_dir)])
    run_quietly(["git", "-C", str(old_dir), "checkout", OLD_COMMIT])
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "300",
        "--movetime-ms", "500",
        "--opponent-cmd", "python -u -m engine",
        "--opponent-cwd", str(old_dir),
        "--sprt", "--elo0", "0", "--elo1", "10",
    ],
    check=True,
)

try:
    publish_results("data: record Stage 6 SPRT (0.6.0 vs 0.5.0)")
except RuntimeError as error:
    print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
    print(error)

In [ ]:
# Rough Elo anchor: Acemi 0.6.0 vs Stockfish with UCI_LimitStrength, UCI_Elo 1320, 500 ms per move.
subprocess.run(
    [
        "python", "-m", "tools.match",
        "--games", "60",
        "--movetime-ms", "500",
        "--uci-elo", "1320",
    ],
    check=True,
)

try:
    publish_results("data: record Stage 6 Stockfish anchor (0.6.0, UCI_Elo 1320)")
except RuntimeError as error:
    print("PUSH FAILED. The records are still in the Kaggle session; run the last cell to retry.")
    print(error)

In [ ]:
# Final push: sends anything the per-match pushes missed. Raises if the push fails.
publish_results("data: record Stage 6 runs")